In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-bureau-credito") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 01:57:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
BASE_PATH = "/app/data"

In [4]:
df_cliente = spark.read.parquet(f"{BASE_PATH}/bronze/cliente")

In [5]:
df_bureau_credito = df_cliente.select("cliente_id") \
    .withColumn("score_bureau", floor(rand() * 701 + 200).cast("int")) \
    .withColumn("qtd_restricoes", floor(rand() * 5).cast("int")) \
    .withColumn("qtd_consultas_90d", floor(rand() * 10).cast("int")) \
    .withColumn("qtd_consultas_365d", floor(rand() * 30).cast("int")) \
    .withColumn(
        "valor_total_dividas",
        round(rand() * 50000, 2)
    ) \
    .withColumn("data_referencia", current_date())

In [6]:
df_bureau_credito.show(10, truncate=False)
df_bureau_credito.printSchema()
df_bureau_credito.count()

+----------+------------+--------------+-----------------+------------------+-------------------+---------------+
|cliente_id|score_bureau|qtd_restricoes|qtd_consultas_90d|qtd_consultas_365d|valor_total_dividas|data_referencia|
+----------+------------+--------------+-----------------+------------------+-------------------+---------------+
|1         |569         |2             |8                |16                |1897.06            |2026-06-24     |
|2         |384         |2             |7                |4                 |47111.8            |2026-06-24     |
|3         |654         |0             |0                |26                |32091.82           |2026-06-24     |
|4         |406         |3             |3                |18                |3821.18            |2026-06-24     |
|5         |459         |4             |9                |8                 |42568.41           |2026-06-24     |
|6         |603         |0             |7                |15                |23223.3    

10000

In [7]:
df_bureau_credito.select(
    min("score_bureau").alias("score_min"),
    max("score_bureau").alias("score_max"),
    avg("score_bureau").alias("score_medio")
).show()

+---------+---------+-----------+
|score_min|score_max|score_medio|
+---------+---------+-----------+
|      200|      900|   550.6135|
+---------+---------+-----------+



In [8]:
df_bureau_credito.write \
    .mode("overwrite") \
    .parquet(f"{BASE_PATH}/bronze/bureau_credito")

In [10]:
spark.stop()